# 03 — Repair READ estimators and validate attribution

This notebook separates three quantities previously conflated as one READ
test: the exact local attribution derivative, a measured small-dose slope, and
the nonlinear full-ablation endpoint. It also fixes weight READ's layer bug by
feeding block `k` with `v[k-1]` and evaluating output orientation against
`v[k]`. Weight magnitude remains unsigned and selection-conditioned.

In [1]:
import json
import os
import sys
from pathlib import Path

ROOT = Path('/home/jovyan/j-space-thoughts')
os.chdir(ROOT)
sys.path.insert(0, str(ROOT))
os.environ['HF_HOME'] = str(Path.home() / '.cache/huggingface')
os.environ['HF_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')
os.environ['HUGGINGFACE_HUB_CACHE'] = str(Path.home() / '.cache/huggingface/hub')

metrics = json.loads((ROOT / 'results/metrics.json').read_text())
repair = metrics['repair_v2']
assert repair['gate_ledger']['g_swap'] == 'PASS'
assert repair['gate_ledger']['g_dir'] in {'PASS', 'DROPPED_MD'}
workspace_layers = repair['stage1']['g_swap']['canonical_configuration']['layers']

from src.jlens_iface import load_published_lens
from src.model_utils import load_model
from src.v2_repair import MODEL_ID

bundle = load_model(MODEL_ID)
lens = load_published_lens(MODEL_ID)
print('READ validation band:', workspace_layers)

Loading weights:   0%|          | 0/339 [00:00<?, ?it/s]

READ validation band: [13, 14, 15, 16, 17, 18, 19, 20, 21, 22, 23, 24]


In [2]:
from src.v2_read import run_stage1d

stage1d = run_stage1d(bundle, lens, workspace_layers=workspace_layers)
a = stage1d['attribution']
w = stage1d['weight_read']
print({
    'READ status': stage1d['status'],
    'primary': stage1d['primary_read'],
    'attribution_role': a['role'],
    'exact_derivative_plumbing': a['plumbing_pass_abs_error_le_0.05'],
    'local_r': a['correlations']['predicted_vs_local_slope']['estimate'],
    'full_alpha1_r': a['correlations']['predicted_vs_full_alpha1_delta']['estimate'],
    'read_strength_r': a['correlations']['read_strength_vs_positive_damage']['estimate'],
    'weight_status': w['status'],
    'weight_above_random_cases': w['n_above_random_cases'],
    'weight_positive_orientation_cases': w['n_positive_orientation_cases'],
})
print('weight criteria:', w['criteria'])

/opt/conda/lib/python3.11/site-packages/torch/autograd/graph.py:825: UserWarning: Flash Attention defaults to a non-deterministic algorithm. To explicitly enable determinism call torch.use_deterministic_algorithms(True, warn_only=False). (Triggered internally at ../aten/src/ATen/native/transformers/cuda/attention_backward.cu:102.)
  return Variable._execution_engine.run_backward(  # Calls into the C++ engine to run the backward pass


{'READ status': 'PASS', 'primary': 'WEIGHT_BASED', 'attribution_role': 'SECONDARY', 'exact_derivative_plumbing': True, 'local_r': 0.17343373711291157, 'full_alpha1_r': 0.061844689450020945, 'read_strength_r': -0.18695128223935775, 'weight_status': 'PASS', 'weight_above_random_cases': 2, 'weight_positive_orientation_cases': 1}
weight criteria: {'all_primary_values_finite': True, 'both_families_above_random_in_at_least_2_of_3_cases': True, 'attribution_weight_rank_rho_nonnegative_for_both_families': True}


In [3]:
from src.v2_read import persist_stage1d

metrics = persist_stage1d(stage1d)
gate = metrics['repair_v2']['gate_ledger']['read_validation']
print('Persisted READ validation:', gate)
assert gate in {'PASS', 'FAIL'}
assert metrics['repair_v2']['gate_ledger']['stage3_science'] == 'PROHIBITED'

Persisted READ validation: PASS


In [4]:
import gc
import torch

del stage1d, metrics, lens, bundle
gc.collect()
torch.cuda.empty_cache()
print('Notebook 03 complete. Controls/G-POS are next only if READ passed.')

Notebook 03 complete. Controls/G-POS are next only if READ passed.
